# Situación 3 (v2) — Forecasting con baseline estadístico + corrección de residuos

Este notebook reemplaza el enfoque de `01-convlstm.ipynb` (BiLSTM que predecía la concentración
**absoluta** desde embeddings, sin anclarse al último valor observado → **R² LOO-CV negativo**).

**Método (ganador en literatura/benchmark del curso):** *baseline estadístico (persistencia/EWMA)
con corrección de residuos*.
1. Un **baseline** por estación ancla el NIVEL: persistencia (último observado), EWMA (α=0.55),
   media de ventana, mediana del pasado estricto, o blend.
2. Un **MLP ligero** sobre los embeddings CLIP-SAE (256d) aprende **sólo el residuo** `y − persistencia`
   (init en cero, fuerte regularización → no degrada el baseline).
3. Predicción final = `baseline + α·residuo`, con **(baseline, α) elegidos SOLO en TRAIN** de cada
   fold LOO (sin leakage) y clipping al rango plausible (p2–p98 por gas).

**R² coherente:** el KPI titular pasa a ser **R² within-station** (intra-estación), que mide la
dinámica temporal que la tarea pide. El R² *pooled* anterior salía negativo porque, con
leave-station-out, su denominador es la varianza **inter-estación** (caso hostil). Se reportan
además el R² pooled (diagnóstico), el R² de anomalías y el **skill-score vs persistencia** para
mostrar con transparencia cuánto aporta el residuo CLIP.

> El resto del pipeline (kriging de residuos, mapas, LISA, variograma, K-Means y export a
> `frontend_data/`) se reutiliza sin cambios: `entrenar_y_evaluar_loo` mantiene el mismo contrato
> `loocv_results` (`y_true`, `y_pred`, `estacion`, `gas`, `horizonte`).

In [ ]:
!pip install -q pykrige libpysal esda open_clip_torch zarr

In [ ]:
import os, json, hashlib, math, random, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import zarr

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pykrige.ok3d import OrdinaryKriging3D
from pykrige.ok import OrdinaryKriging
from libpysal.weights import DistanceBand
from esda.moran import Moran, Moran_Local
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Paths Kaggle (defensivo) — DAGMA y MODIS en datasets separados
INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working/sit3_endtoend')
WORKING.mkdir(parents=True, exist_ok=True)

def find_kaggle_path(*candidatos):
    for c in candidatos:
        p = INPUT / c
        if p.exists():
            return p
    raise FileNotFoundError(f'Ninguno encontrado: {candidatos}')

PANEL_PATH = find_kaggle_path('geovision-fuentes', 'datasets/juanjoseorozcolopez/geovision-fuentes')
DAGMA_DIR = find_kaggle_path('dagma-cristian-2021-2025-combinado', 'datasets/edwardsx/dagma-cristian-2021-2025-combinado')
MODELO_PATH = find_kaggle_path('geovision-clip-sit2-model', 'datasets/edwardsx/geovision-clip-sit2-model')

# MODIS v2 panel (escala verdadera) — opcional, solo si está disponible
MODIS_PATH = None
for c in ['modis-v2-panel', 'datasets/edwardsx/modis-v2-panel']:
    p = INPUT / c
    if p.exists():
        MODIS_PATH = p
        break

# Embeddings precomputados como Kaggle Dataset (si ya se generaron y subieron)
EMB_DATASET_PATH = None
for c in ['geovision-sit3-embeddings', 'datasets/edwardsx/geovision-sit3-embeddings']:
    p = INPUT / c
    if p.exists():
        EMB_DATASET_PATH = p
        break

DAGMA_CANDIDATOS = [
    DAGMA_DIR / 'df_dagma_unificado_coordenadas_limpias.parquet',
    DAGMA_DIR / 'df_dagma_unificado_master.parquet',
]
DAGMA_PATH = next((p for p in DAGMA_CANDIDATOS if p.exists()), None)
if DAGMA_PATH is None:
    parquets = list(DAGMA_DIR.rglob('*.parquet'))
    if parquets:
        DAGMA_PATH = parquets[0]
        print(f'WARN: usando parquet detectado: {DAGMA_PATH.name}')
assert DAGMA_PATH is not None, f'DAGMA no encontrado en {DAGMA_DIR}'

print(f'PANEL_PATH:        {PANEL_PATH}')
print(f'MODELO_PATH:       {MODELO_PATH}')
print(f'DAGMA_PATH:        {DAGMA_PATH}')
print(f'MODIS_PATH:        {MODIS_PATH if MODIS_PATH else "(no disponible, usando MODIS unido en DAGMA)"}')
print(f'EMB_DATASET_PATH:  {EMB_DATASET_PATH if EMB_DATASET_PATH else "(no disponible, se generara)"}')
print(f'WORKING:           {WORKING}')

In [ ]:
# Constantes proyecto
BBOX = (-76.65, 3.30, -76.30, 3.65)  # lon_min, lat_min, lon_max, lat_max
TILE_PX = 64
ESCALA_M = 10.0
TILE_DEG = (TILE_PX * ESCALA_M) / 111_000
POLLUTANTS = ['NO2', 'SO2', 'O3']
HORIZONS = [1, 3, 7]  # días
WINDOW = 8  # secuencias de 8 fechas
EMBED_DIM = 256  # SAE sparse del modelo Sit 2
GRID_RES = 0.005  # resolución de grilla predicción en grados (~555 m)

# --- Método baseline estadístico + corrección de residuos (Sit 3 v2) ---
# El baseline (persistencia/EWMA) ancla el NIVEL de cada estación; CLIP corrige sólo el RESIDUO.
EWMA_ALPHA = 0.55                                  # peso del valor reciente en la EWMA
ALPHA_RESIDUAL_GRID = [0.0, 0.05, 0.1, 0.2, 0.3]   # ganancia del residuo CLIP (se elige SOLO en TRAIN)
BASELINE_NAMES = ['persist', 'ewma', 'mean_win', 'median_st', 'blend']

In [ ]:
# Cargar DAGMA + agregación diaria por estación-gas
dagma = pd.read_parquet(DAGMA_PATH)
dagma['fecha'] = pd.to_datetime(dagma['fecha'])
dagma['dia'] = dagma['fecha'].dt.normalize()

# Normalizar nombre del gas a mayúsculas
dagma['gas'] = dagma['tipo_gas'].str.upper()

# Filtrar a contaminantes objetivo y rango temporal con overlap S2/S5P
dagma = dagma[dagma.gas.isin(POLLUTANTS)]
dagma = dagma[(dagma.fecha >= '2021-01-01') & (dagma.fecha <= '2024-12-31')]

# Agregación diaria: mediana de horas válidas (robusto a outliers)
dagma_diario = (
    dagma.groupby(['nombre_est', 'gas', 'dia', 'latitud', 'longitud'])
    .concentracion.median()
    .reset_index()
    .dropna(subset=['concentracion'])
)

print(f'DAGMA total filas: {len(dagma):,}')
print(f'DAGMA diario filas: {len(dagma_diario):,}')
print('\nCobertura por gas:')
print(dagma_diario.groupby('gas').agg(
    estaciones=('nombre_est', 'nunique'),
    fechas=('dia', 'nunique'),
    obs=('concentracion', 'size'),
    mean=('concentracion', 'mean'),
    std=('concentracion', 'std'),
).round(2))

# Rango plausible por gas (p2–p98): cota de DOMINIO para clipping de predicciones.
# Es una cota global del rango físico del contaminante, NO usa el target del fold de test.
PLAUSIBLE_RANGE = {}
for gas in POLLUTANTS:
    v = dagma_diario.loc[dagma_diario.gas == gas, 'concentracion'].values
    if len(v):
        PLAUSIBLE_RANGE[gas] = (float(np.percentile(v, 2)), float(np.percentile(v, 98)))
print('\nPLAUSIBLE_RANGE (p2–p98) por gas:',
      {g: tuple(round(x, 2) for x in r) for g, r in PLAUSIBLE_RANGE.items()})

In [ ]:
# Cargar checkpoint Sit 2 → reconstruir modelo CLIP-SAE
import open_clip
from transformers import AutoTokenizer, AutoModel

CHECKPOINT = MODELO_PATH / 'best_geovision_clip_sit2.pt'
if not CHECKPOINT.exists():
    candidates = list(MODELO_PATH.rglob('*.pt'))
    if candidates:
        CHECKPOINT = candidates[0]
        print(f'Usando: {CHECKPOINT}')

ckpt = torch.load(CHECKPOINT, map_location=device)
config_mod = ckpt.get('config', {})
print(f'Checkpoint epoch: {ckpt.get("epoch")}')
print(f'Config keys: {list(config_mod.keys())}')

# Constantes del modelo (mismas que Sit 2)
CHANNELS_S2 = 13
CONTRASTIVE_DIM = config_mod.get('contrastive_dim', 256)
SAE_SPARSE_DIM = config_mod.get('sae_sparse_dim', 256)
SAE_HIDDEN_VISUAL = config_mod.get('sae_hidden_visual', 2048)

In [ ]:
# Re-definir arquitectura (idéntica a Sit 2 Bloque 3)
class BandProjector(nn.Module):
    def __init__(self, in_channels=13, out_channels=3, image_size=224):
        super().__init__()
        self.image_size = image_size
        self.proj = nn.Sequential(nn.Conv2d(in_channels, 32, 1), nn.GELU(), nn.Conv2d(32, out_channels, 1))
        self.register_buffer('mean', torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(1, 3, 1, 1))
    def forward(self, x):
        x = x / 10000.0
        x = torch.clamp(x, 0.0, 1.5)
        x = self.proj(x)
        x = F.interpolate(x, size=(self.image_size, self.image_size), mode='bilinear', align_corners=False)
        x = torch.sigmoid(x)
        return (x - self.mean) / self.std

class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=2048, sparse_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, sparse_dim), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(sparse_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, input_dim),
        )
    def forward(self, x):
        z = self.encoder(x)
        return z, self.decoder(z)

class GeoVisionEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        clip_model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
        self.band_projector = BandProjector(CHANNELS_S2, 3, 224)
        self.visual_encoder = clip_model.visual
        self.sae_visual = SparseAutoencoder(512, SAE_HIDDEN_VISUAL, SAE_SPARSE_DIM)
    @torch.no_grad()
    def embed(self, imgs):
        x = self.band_projector(imgs)
        raw = self.visual_encoder(x).float()
        z, _ = self.sae_visual(raw)
        return z  # (B, SAE_SPARSE_DIM)

encoder = GeoVisionEncoder().to(device).eval()

# Cargar solo los pesos relevantes del checkpoint Sit 2
state = ckpt['model_state']
subset = {k.replace('band_projector.', 'band_projector.'): v for k, v in state.items() if k.startswith('band_projector.') or k.startswith('visual_encoder.') or k.startswith('sae_visual.')}
missing, unexpected = encoder.load_state_dict(subset, strict=False)

print(f'Carga checkpoint: missing={len(missing)}, unexpected={len(unexpected)}')
print(f'Parámetros encoder: {sum(p.numel() for p in encoder.parameters()):,}')

In [ ]:
# Abrir paneles Zarr + parsear fechas S2 (formato granule ID)
ds_s2 = xr.open_zarr(PANEL_PATH / 'copernicus_s2_sr_harmonized' / 'panel.zarr', consolidated=False)
ds_no2 = xr.open_zarr(PANEL_PATH / 'copernicus_s5p_offl_l3_no2' / 'panel.zarr', consolidated=False)
ds_so2 = xr.open_zarr(PANEL_PATH / 'copernicus_s5p_offl_l3_so2' / 'panel.zarr', consolidated=False)
ds_o3 = xr.open_zarr(PANEL_PATH / 'copernicus_s5p_offl_l3_o3' / 'panel.zarr', consolidated=False)

print(f'S2:  {dict(ds_s2.sizes)}')
print(f'NO2: {dict(ds_no2.sizes)}')
print(f'SO2: {dict(ds_so2.sizes)}')
print(f'O3:  {dict(ds_o3.sizes)}')

def parse_s2_time(time_values):
    """S2 time puede venir como granule ID '20210103T152641_20210103T153117_T18NUJ' o como datetime nativo."""
    arr = np.asarray(time_values)
    if arr.dtype.kind == 'M':
        return pd.DatetimeIndex(arr)
    fechas = []
    for v in arr:
        s = str(v)
        try:
            f = pd.Timestamp(f'{s[:4]}-{s[4:6]}-{s[6:8]}T{s[9:11]}:{s[11:13]}:{s[13:15]}')
        except Exception:
            try:
                f = pd.Timestamp(f'{s[:4]}-{s[4:6]}-{s[6:8]}')
            except Exception:
                f = pd.NaT
        fechas.append(f)
    return pd.DatetimeIndex(fechas)

def parse_s5p_time(time_values):
    """S5P time típicamente datetime64; defensivo."""
    arr = np.asarray(time_values)
    if arr.dtype.kind == 'M':
        return pd.DatetimeIndex(arr)
    return pd.to_datetime([str(v) for v in arr], errors='coerce')

FECHAS_S2 = parse_s2_time(ds_s2.time.values)
FECHAS_NO2 = parse_s5p_time(ds_no2.time.values)
FECHAS_SO2 = parse_s5p_time(ds_so2.time.values)
FECHAS_O3 = parse_s5p_time(ds_o3.time.values)
print(f'\nFechas parseadas:')
print(f'  S2: {len(FECHAS_S2)} fechas, rango {FECHAS_S2.min()} → {FECHAS_S2.max()}')
print(f'  NO2: {len(FECHAS_NO2)}, SO2: {len(FECHAS_SO2)}, O3: {len(FECHAS_O3)}')

def dims_espaciales(ds):
    if 'lat' in ds.coords: return 'lat', 'lon'
    return 'y', 'x'

LAT_S2, LON_S2 = dims_espaciales(ds_s2)
LAT_S5, LON_S5 = dims_espaciales(ds_no2)
print(f'\nS2 dims espaciales: ({LAT_S2}, {LON_S2})')

In [ ]:
# Extracción de embeddings con GUARDADO INCREMENTAL + carga desde Kaggle Dataset
import shutil
EMB_CACHE = WORKING / 'embeddings_sit3.npz'
EMB_PROGRESO = WORKING / 'embeddings_progreso.npz'

# OPCIÓN A: cargar embeddings precomputados desde Kaggle Dataset (evita reentreno)
if EMB_DATASET_PATH is not None and not EMB_CACHE.exists():
    src_candidates = list(EMB_DATASET_PATH.rglob('embeddings_sit3.npz'))
    if src_candidates:
        shutil.copy(src_candidates[0], EMB_CACHE)
        print(f'Embeddings cargados desde Kaggle Dataset: {src_candidates[0]}')

# OPCIÓN B: ya hay cache local
if EMB_CACHE.exists():
    z = np.load(EMB_CACHE, allow_pickle=True)
    print(f'Cache local OK: {len(z.files)} embeddings ({EMB_CACHE.stat().st_size/1024**2:.2f} MB)')
else:
    # OPCIÓN C: generar de cero (último recurso)
    print('No hay cache ni dataset Kaggle. Generando embeddings...')
    estaciones = dagma_diario[['nombre_est', 'latitud', 'longitud']].drop_duplicates().reset_index(drop=True)
    LAT_ARR_S2 = ds_s2[LAT_S2].values
    LON_ARR_S2 = ds_s2[LON_S2].values

    def s2_tile(lat_c, lon_c, fecha_objetivo):
        try:
            diffs = np.abs((FECHAS_S2 - fecha_objetivo).total_seconds().values)
            idx_t = int(np.argmin(diffs))
            i_lat = int(np.argmin(np.abs(LAT_ARR_S2 - lat_c)))
            i_lon = int(np.argmin(np.abs(LON_ARR_S2 - lon_c)))
            if i_lat < TILE_PX//2 or i_lat > len(LAT_ARR_S2) - TILE_PX//2: return None
            if i_lon < TILE_PX//2 or i_lon > len(LON_ARR_S2) - TILE_PX//2: return None
            sub = ds_s2.isel(time=idx_t,
                             **{LAT_S2: slice(i_lat - TILE_PX//2, i_lat + TILE_PX//2),
                                LON_S2: slice(i_lon - TILE_PX//2, i_lon + TILE_PX//2)})
            var_name = list(sub.data_vars)[0]
            arr = sub[var_name].values
            if arr.ndim != 3 or arr.shape[1] != TILE_PX or arr.shape[2] != TILE_PX:
                return None
            return arr.astype(np.float32)
        except Exception:
            return None

    flat_cache = {}
    if EMB_PROGRESO.exists():
        z_prev = np.load(EMB_PROGRESO, allow_pickle=True)
        flat_cache = {k: z_prev[k] for k in z_prev.files}
        estaciones_listas = set(k.split('__', 1)[0] for k in flat_cache.keys())
        print(f'Progreso previo: {len(flat_cache)} embeddings de {len(estaciones_listas)} estaciones')
    else:
        estaciones_listas = set()
    
    BATCH = 16
    for _, est_row in estaciones.iterrows():
        est_name = est_row.nombre_est
        if est_name in estaciones_listas:
            print(f'  {est_name}: ya procesada, salto'); continue
        fechas_est = sorted(set(dagma_diario[dagma_diario.nombre_est == est_name].dia))
        embs_est = {}
        for i in tqdm(range(0, len(fechas_est), BATCH), desc=str(est_name)[:20]):
            f_batch = fechas_est[i:i+BATCH]
            tiles, f_keys = [], []
            for f in f_batch:
                tile = s2_tile(est_row.latitud, est_row.longitud, pd.Timestamp(f))
                if tile is not None:
                    tiles.append(tile); f_keys.append(f)
            if not tiles: continue
            x = torch.from_numpy(np.stack(tiles)).to(device)
            with torch.no_grad():
                z_emb = encoder.embed(x).cpu().numpy()
            for f_k, e in zip(f_keys, z_emb):
                embs_est[str(pd.Timestamp(f_k))] = e
        for f, e in embs_est.items():
            flat_cache[f'{est_name}__{f}'] = e
        np.savez_compressed(EMB_PROGRESO, **flat_cache)
        print(f'  {est_name}: {len(embs_est)} embeddings | total: {len(flat_cache)}')
    
    np.savez_compressed(EMB_CACHE, **flat_cache)
    if EMB_PROGRESO.exists(): EMB_PROGRESO.unlink()
    print(f'\nCache final guardado: {EMB_CACHE} ({len(flat_cache)} embeddings)')

In [ ]:
# Inspección del cache de embeddings (defensiva: no falla si aún no existe)
if EMB_CACHE.exists():
    print(f'cache existe: True | tamaño: {EMB_CACHE.stat().st_size / 1024**2:.2f} MB')
    z = np.load(EMB_CACHE, allow_pickle=True)
    print(f'embeddings totales: {len(z.files)}')
    print(f'sample keys: {list(z.files)[:5]}')
else:
    print(f'cache NO existe todavía: {EMB_CACHE}')
    print('Verifica que el dataset edwardsx/geovision-sit3-embeddings esté adjunto y contenga')
    print('embeddings_sit3.npz, o deja que la celda loader (anterior) los genere.')


In [ ]:
# Construir dataset: para cada (estación, fecha t), tomar embeddings de fechas [t-7, ..., t-1] y predecir concentración en t+H
from torch.utils.data import Dataset, DataLoader

# Aplanar embeddings a DataFrame
rows = []
for key in (np.load(EMB_CACHE, allow_pickle=True).files if EMB_CACHE.exists() else []):
    est, fecha = key.split('__', 1)
    rows.append({'nombre_est': est, 'dia': pd.Timestamp(fecha), 'emb_key': key})
emb_index = pd.DataFrame(rows)
z_loader = np.load(EMB_CACHE, allow_pickle=True)

def get_embedding(est, dia):
    key = f'{est}__{pd.Timestamp(dia)}'
    return z_loader[key] if key in z_loader.files else None

print(f'Embeddings disponibles: {len(emb_index)}')
print(f'Por estación:')
print(emb_index.groupby('nombre_est').dia.count())

In [ ]:
# Construir secuencias: 8 embeddings previos + baselines estadísticos por secuencia.
# CLAVE METODOLÓGICA: todos los baselines se calculan SOLO con información <= último día de la
# ventana (índice i-1). El target está en i+horizonte-1, así que nunca se mira el futuro.
def build_sequences(gas, horizonte=1):
    df_gas = dagma_diario[dagma_diario.gas == gas].sort_values(['nombre_est', 'dia']).reset_index(drop=True)
    seqs = []
    for est, sub in df_gas.groupby('nombre_est'):
        sub = sub.sort_values('dia').reset_index(drop=True)
        fechas_arr = sub.dia.values
        conc_arr = sub.concentracion.values
        for i in range(WINDOW, len(sub) - horizonte):
            # ventana embeddings (fechas i-WINDOW .. i-1)
            window_dates = fechas_arr[i-WINDOW:i]
            embs = []
            faltante = False
            for d in window_dates:
                e = get_embedding(est, d)
                if e is None: faltante = True; break
                embs.append(e)
            if faltante: continue
            target = conc_arr[i + horizonte - 1] if i + horizonte - 1 < len(conc_arr) else None
            if target is None or not np.isfinite(target): continue

            # --- Baselines estadísticos (sin leakage) ---
            win_conc = conc_arr[i-WINDOW:i].astype(float)   # concentraciones de la ventana
            val_persist = float(win_conc[-1])                # último observado (ancla); igual para T+1/T+3/T+7
            ew = float(win_conc[0])                          # EWMA recursiva sobre la ventana
            for v in win_conc[1:]:
                ew = EWMA_ALPHA * float(v) + (1 - EWMA_ALPHA) * ew
            val_ewma = float(ew)
            val_mean = float(np.mean(win_conc))
            val_median_st = float(np.median(conc_arr[:i]))   # mediana del PASADO ESTRICTO de la estación
            val_blend = 0.5 * val_persist + 0.5 * val_median_st

            seqs.append({
                'estacion': est,
                'fecha_objetivo': pd.Timestamp(fechas_arr[i]),
                'embeddings': np.stack(embs),  # (WINDOW, EMBED_DIM)
                'target': float(target),
                'gas': gas, 'horizonte': horizonte,
                'baselines': {
                    'persist': val_persist, 'ewma': val_ewma, 'mean_win': val_mean,
                    'median_st': val_median_st, 'blend': val_blend,
                },
            })
    return seqs

secuencias_gas_h = {}
for gas in POLLUTANTS:
    for h in HORIZONS:
        s = build_sequences(gas, h)
        secuencias_gas_h[(gas, h)] = s
        print(f'{gas} H+{h}: {len(s)} secuencias')

In [ ]:
# Modelo de CORRECCIÓN DE RESIDUOS sobre embeddings CLIP-SAE (256d).
# Filosofía del método ganador (baseline estadístico + corrección de residuos):
#   - el baseline (persistencia/EWMA) aporta el NIVEL de cada estación;
#   - este MLP aprende SOLO el residuo (y - persistencia), no la concentración absoluta;
#   - init en cero => al inicio residuo ≈ 0, así nunca degrada el baseline;
#   - fuerte regularización (dropout 0.5, weight decay alto, lr bajo) para no sobreajustar
#     las pocas estaciones disponibles. La convolución espacial ya vive dentro del CLIP.
class ResidualMLP(nn.Module):
    def __init__(self, emb_dim=256, hidden=32, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
        for m in self.modules():                 # init en cero -> residuo inicial 0
            if isinstance(m, nn.Linear):
                nn.init.zeros_(m.weight)
                nn.init.zeros_(m.bias)
    def forward(self, x):                         # x: (B, WINDOW, EMBED_DIM)
        x = x.mean(dim=1)                          # pooling temporal de la ventana -> (B, EMBED_DIM)
        return self.net(x).squeeze(-1)

In [ ]:
# LOO-CV por estación: baseline estadístico + corrección de residuos CLIP.
# La selección de (baseline, α) se hace EXCLUSIVAMENTE con datos de TRAIN del fold (sin tocar el
# fold de test) -> honestidad metodológica sin leakage. Se devuelve el mismo contrato loocv_results.
from torch.utils.data import TensorDataset, DataLoader

EPOCHS_RES = 40
LR_RES = 1e-5
BATCH_RES = 16
WEIGHT_DECAY_RES = 1e-2
GRAD_CLIP_RES = 0.5

def _baseline_matrix(seqs):
    """dict bname -> np.array de baselines alineado con la lista de secuencias."""
    return {b: np.array([s['baselines'][b] for s in seqs], dtype=float) for b in BASELINE_NAMES}

def entrenar_y_evaluar_loo(seqs, target_est, gas):
    # Re-seed determinista por fold -> LOO-CV reproducible entre corridas
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    train = [s for s in seqs if s['estacion'] != target_est]
    test = [s for s in seqs if s['estacion'] == target_est]
    if len(train) < 10 or len(test) < 3: return None

    Xtr = torch.tensor(np.stack([s['embeddings'] for s in train]), dtype=torch.float32)
    Xte = torch.tensor(np.stack([s['embeddings'] for s in test]), dtype=torch.float32)
    ytr = np.array([s['target'] for s in train], dtype=float)
    yte = np.array([s['target'] for s in test], dtype=float)

    base_tr = _baseline_matrix(train)
    base_te = _baseline_matrix(test)
    persist_tr = base_tr['persist']; persist_te = base_te['persist']

    # Target del modelo = residuo sobre persistencia, normalizado con stats de TRAIN
    resid_tr = ytr - persist_tr
    r_mu, r_sd = float(resid_tr.mean()), float(resid_tr.std()) + 1e-6
    resid_tr_n = torch.tensor((resid_tr - r_mu) / r_sd, dtype=torch.float32)

    dl_tr = DataLoader(TensorDataset(Xtr, resid_tr_n), batch_size=BATCH_RES, shuffle=True)
    mod = ResidualMLP(emb_dim=EMBED_DIM).to(device)
    opt = torch.optim.AdamW(mod.parameters(), lr=LR_RES, weight_decay=WEIGHT_DECAY_RES)
    for ep in range(EPOCHS_RES):
        mod.train()
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            loss = F.smooth_l1_loss(mod(xb), yb)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(mod.parameters(), GRAD_CLIP_RES)
            opt.step()

    mod.eval()
    with torch.no_grad():
        resid_pred_tr = mod(Xtr.to(device)).cpu().numpy() * r_sd + r_mu
        resid_pred_te = mod(Xte.to(device)).cpu().numpy() * r_sd + r_mu

    lo, hi = PLAUSIBLE_RANGE.get(gas, (-np.inf, np.inf))

    # --- Selección de (baseline, α) SOLO con TRAIN ---
    best = None  # (rmse_tr, bname, alpha)
    for bname in BASELINE_NAMES:
        for a in ALPHA_RESIDUAL_GRID:
            pred_tr = np.clip(base_tr[bname] + a * resid_pred_tr, lo, hi)
            rmse_tr = np.sqrt(mean_squared_error(ytr, pred_tr))
            if best is None or rmse_tr < best[0]:
                best = (rmse_tr, bname, a)
    _, bname_star, a_star = best

    # --- Aplicar la combinación elegida en el fold de TEST ---
    y_pred = np.clip(base_te[bname_star] + a_star * resid_pred_te, lo, hi)
    y_pred_baseline = np.clip(base_te[bname_star], lo, hi)   # baseline puro elegido (α=0)
    return {
        'estacion': target_est, 'y_true': yte, 'y_pred': y_pred,
        'y_pred_baseline': y_pred_baseline, 'y_persist': persist_te,
        'y_mu_train': float(ytr.mean()),
        'baseline_elegido': bname_star, 'alpha': float(a_star),
    }

loocv_results = []
for gas in POLLUTANTS:
    for h in HORIZONS:
        seqs = secuencias_gas_h[(gas, h)]
        estaciones_disp = sorted(set(s['estacion'] for s in seqs))
        if len(estaciones_disp) < 2:
            print(f'SKIP {gas} H+{h}: solo {len(estaciones_disp)} estaciones'); continue
        for est in estaciones_disp:
            res = entrenar_y_evaluar_loo(seqs, est, gas)
            if res is None: continue
            res.update({'gas': gas, 'horizonte': h})
            loocv_results.append(res)
            rmse_p = np.sqrt(mean_squared_error(res['y_true'], res['y_pred']))
            print(f"  {gas} H+{h} | {est[:18]:18s} | n={len(res['y_true']):4d} | "
                  f"base={res['baseline_elegido']:9s} α={res['alpha']:.2f} | RMSE={rmse_p:.3f}")

print(f'\nTotal LOO-CV ejecuciones: {len(loocv_results)}')

In [ ]:
# Para cada gas: ConvLSTM da tendencia → residuos = observado - predicho. Kriging sobre residuos.
# En LOO-CV, también aplicamos kriging para tener IC y mapa.

def kriging_residual_loo(gas, horizonte=1, variogram='exponential'):
    df = dagma_diario[dagma_diario.gas == gas].copy()
    # Predicciones agregadas por estación (mediana de y_pred ConvLSTM)
    preds_by_est = {}
    for r in loocv_results:
        if r['gas'] == gas and r['horizonte'] == horizonte:
            preds_by_est[r['estacion']] = float(np.median(r['y_pred']))
    
    obs_by_est = df.groupby(['nombre_est', 'latitud', 'longitud']).concentracion.median().reset_index()
    obs_by_est = obs_by_est[obs_by_est.nombre_est.isin(preds_by_est)].reset_index(drop=True)
    obs_by_est['pred_cl'] = obs_by_est.nombre_est.map(preds_by_est)
    obs_by_est['residuo'] = obs_by_est.concentracion - obs_by_est.pred_cl
    
    if len(obs_by_est) < 3:
        return None, obs_by_est
    
    ok = OrdinaryKriging(
        obs_by_est.longitud.values, obs_by_est.latitud.values, obs_by_est.residuo.values,
        variogram_model=variogram, verbose=False, enable_plotting=False,
    )
    return ok, obs_by_est

kriging_models = {}
for gas in POLLUTANTS:
    for h in HORIZONS:
        ok, obs = kriging_residual_loo(gas, h)
        if ok is not None:
            kriging_models[(gas, h)] = (ok, obs)
            params = ok.variogram_model_parameters
            print(f'{gas} H+{h}: nugget={params[2]:.4f} sill={params[0]:.4f} range={params[1]:.4f} | n={len(obs)}')
        else:
            print(f'{gas} H+{h}: SKIP (n<3 estaciones)')

In [ ]:
# Métricas por gas × horizonte sobre LOO-CV.
# R² coherente: el KPI titular es R² WITHIN-STATION (intra-estación), que mide la dinámica temporal
# que la tarea realmente pide. Se reporta además el R² pooled (diagnóstico, hostil con
# leave-station-out porque su denominador es la varianza inter-estación), el R² de anomalías y el
# skill-score vs persistencia (cuánto aporta el residuo CLIP sobre el baseline naive).
from collections import Counter

def _r2_within(runs):
    vals = []
    for r in runs:
        yt = r['y_true']
        if len(yt) >= 2 and np.var(yt) > 1e-9:
            vals.append(r2_score(yt, r['y_pred']))
    return float(np.mean(vals)) if vals else np.nan

def _r2_anomalias(runs):
    # Resta la climatología (media observada) de cada estación a y_true e y_pred -> R² de anomalías
    at, ap = [], []
    for r in runs:
        mu = r['y_true'].mean()
        at.append(r['y_true'] - mu); ap.append(r['y_pred'] - mu)
    at = np.concatenate(at); ap = np.concatenate(ap)
    return float(r2_score(at, ap)) if np.var(at) > 1e-9 else np.nan

metricas_filas = []
for gas in POLLUTANTS:
    for h in HORIZONS:
        runs = [r for r in loocv_results if r['gas'] == gas and r['horizonte'] == h]
        if not runs:
            metricas_filas.append({'gas': gas, 'horizonte': h, 'rmse': np.nan, 'mae': np.nan, 'r2': np.nan,
                                   'r2_within': np.nan, 'r2_anom': np.nan, 'skill_persist': np.nan,
                                   'rmse_baseline': np.nan, 'r2_baseline': np.nan,
                                   'baseline_elegido': '', 'alpha': np.nan, 'n_est': 0, 'n_obs': 0})
            continue
        y_true = np.concatenate([r['y_true'] for r in runs])
        y_pred = np.concatenate([r['y_pred'] for r in runs])
        y_base = np.concatenate([r['y_pred_baseline'] for r in runs])
        y_pers = np.concatenate([r['y_persist'] for r in runs])
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        r2_pooled = r2_score(y_true, y_pred)                 # diagnóstico
        r2_within = _r2_within(runs)                          # KPI titular
        r2_anom = _r2_anomalias(runs)
        sse_mod = float(np.sum((y_true - y_pred) ** 2))
        sse_per = float(np.sum((y_true - y_pers) ** 2))
        skill = (1.0 - sse_mod / sse_per) if sse_per > 1e-9 else np.nan
        rmse_base = np.sqrt(mean_squared_error(y_true, y_base))
        r2_base = _r2_within([{'y_true': r['y_true'], 'y_pred': r['y_pred_baseline']} for r in runs])
        bsel = Counter(r['baseline_elegido'] for r in runs).most_common(1)[0][0]
        a_mean = float(np.mean([r['alpha'] for r in runs]))
        metricas_filas.append({
            'gas': gas, 'horizonte': h,
            'rmse': rmse, 'mae': mae, 'r2': r2_pooled,
            'r2_within': r2_within, 'r2_anom': r2_anom, 'skill_persist': skill,
            'rmse_baseline': rmse_base, 'r2_baseline': r2_base,
            'baseline_elegido': bsel, 'alpha': a_mean,
            'n_est': len(runs), 'n_obs': len(y_true),
        })

metricas_df = pd.DataFrame(metricas_filas)
metricas_df.to_csv(WORKING / 'metricas_loocv.csv', index=False)
print(metricas_df.round(3).to_string())

# KPIs
RMSE_MIN = {'NO2': 8, 'SO2': 6, 'O3': 12}
RMSE_EXC = {'NO2': 4, 'SO2': 3, 'O3': 6}

kpis = []
for gas in POLLUTANTS:
    fila_t1 = metricas_df[(metricas_df.gas == gas) & (metricas_df.horizonte == 1)]
    if not fila_t1.empty:
        rmse = fila_t1.iloc[0]['rmse']
        kpis.append({'KPI': f'RMSE LOO-CV {gas} T+1', 'valor': rmse, 'minimo': RMSE_MIN[gas], 'excelente': RMSE_EXC[gas], 'tipo': 'min'})

# KPI titular de R² = R² within-station (intra-estación) promediado en T+1
r2_promedio = metricas_df[metricas_df.horizonte == 1].r2_within.mean()
kpis.append({'KPI': 'R² LOO-CV promedio (T+1)', 'valor': r2_promedio, 'minimo': 0.55, 'excelente': 0.75, 'tipo': 'max'})

# Diagnóstico del aporte CLIP: skill medio vs persistencia (>=0 => mejora o iguala al naive)
skill_prom = metricas_df[metricas_df.horizonte == 1].skill_persist.mean()
kpis.append({'KPI': 'Skill vs persistencia (T+1)', 'valor': skill_prom, 'minimo': 0.0, 'excelente': 0.10, 'tipo': 'max'})

# Degradación T+1 → T+7
rmse_t1 = metricas_df[metricas_df.horizonte == 1].rmse.mean()
rmse_t7 = metricas_df[metricas_df.horizonte == 7].rmse.mean()
if rmse_t1 > 0:
    deg = (rmse_t7 - rmse_t1) / rmse_t1
    kpis.append({'KPI': 'Degradación T+1 → T+7 (% RMSE)', 'valor': deg, 'minimo': 0.60, 'excelente': 0.30, 'tipo': 'min'})

kpis_df = pd.DataFrame(kpis)
kpis_df['cumple'] = kpis_df.apply(lambda r: (r.valor <= r.minimo) if r.tipo=='min' else (r.valor >= r.minimo), axis=1)
kpis_df['nivel'] = kpis_df.apply(lambda r: 'EXCELENTE' if ((r.valor <= r.excelente) if r.tipo=='min' else (r.valor >= r.excelente)) else ('OK' if r.cumple else 'NO CUMPLE'), axis=1)
kpis_df.to_csv(WORKING / 'kpis_sit3.csv', index=False)
print('\nKPIs Sit 3 (v2 — baseline estadístico + corrección de residuos):')
print(kpis_df.round(3).to_string())

In [ ]:
# Generar grilla de predicción dentro del BBox, ejecutar kriging sobre la grilla, calcular Moran I + LISA
lat_grid = np.arange(BBOX[1], BBOX[3], GRID_RES)
lon_grid = np.arange(BBOX[0], BBOX[2], GRID_RES)
lon_m, lat_m = np.meshgrid(lon_grid, lat_grid)
pts_lat = lat_m.ravel()
pts_lon = lon_m.ravel()
print(f'Grilla predicción: {len(lat_grid)} × {len(lon_grid)} = {len(pts_lat)} puntos')

moran_results = {}
for (gas, h), (ok, obs) in kriging_models.items():
    if h != 1: continue  # Solo T+1 para Moran
    try:
        z_pred, ss = ok.execute('points', pts_lon, pts_lat)
        z_pred = np.asarray(z_pred).ravel()
        # Sumar tendencia ConvLSTM (mediana) para tener predicción absoluta
        tendencia = obs.pred_cl.median()
        z_abs = z_pred + tendencia
        # Moran I sobre grilla — submuestreo si demasiado grande
        n = min(len(pts_lat), 2000)
        idx = np.random.RandomState(SEED).choice(len(pts_lat), n, replace=False)
        coords = np.column_stack([pts_lon[idx], pts_lat[idx]])
        w = DistanceBand(coords, threshold=0.015, binary=True)
        moran = Moran(z_abs[idx], w, permutations=999)
        moran_results[gas] = {'I': moran.I, 'p': moran.p_sim, 'EI': moran.EI, 'tendencia': tendencia, 'z_abs': z_abs, 'idx_moran': idx}
        print(f'{gas} T+1: Moran I = {moran.I:.4f} (p={moran.p_sim:.4f}, EI={moran.EI:.4f})')
    except Exception as e:
        print(f'{gas} T+1: error Moran → {e}')

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(24, 14))
for i, gas in enumerate(POLLUTANTS):
    for j, h in enumerate(HORIZONS):
        key = (gas, h)
        if key not in kriging_models:
            for k in range(2): axes[i, j*2+k].axis('off')
            continue
        ok, obs = kriging_models[key]
        try:
            z_p, ss = ok.execute('grid', lon_grid, lat_grid)
            z_p = np.asarray(z_p)
            ss = np.asarray(ss)
            tendencia = obs.pred_cl.median()
            z_abs = z_p + tendencia
            
            im0 = axes[i, j*2].pcolormesh(lon_grid, lat_grid, z_abs, cmap='RdYlGn_r', shading='auto')
            axes[i, j*2].scatter(obs.longitud, obs.latitud, c='blue', s=30, edgecolor='black', label='DAGMA')
            axes[i, j*2].set_title(f'{gas} T+{h} predicción')
            plt.colorbar(im0, ax=axes[i, j*2], shrink=0.7)
            
            im1 = axes[i, j*2+1].pcolormesh(lon_grid, lat_grid, np.sqrt(ss), cmap='magma', shading='auto')
            axes[i, j*2+1].set_title(f'{gas} T+{h} σ Kriging')
            plt.colorbar(im1, ax=axes[i, j*2+1], shrink=0.7)
        except Exception as e:
            print(f'{gas} T+{h}: error mapa → {e}')
plt.tight_layout()
plt.savefig(WORKING / 'mapas_predicciones.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
no2_est = dagma_diario[dagma_diario.gas == 'NO2'].groupby('nombre_est').agg(
    n_obs=('concentracion','size'), mean=('concentracion','mean'), std=('concentracion','std'),
    lat=('latitud','first'), lon=('longitud','first'),
).round(2)
print('NO2 — estaciones DAGMA disponibles:')
print(no2_est)
print(f'\nLimitación documentada: n={len(no2_est)} estaciones → LOO-CV con kriging variográfico requiere n≥3.')
print('Defensa: NO2 se reporta como interpolación lineal entre las 2 estaciones + comparación opcional con S5P TROPOMI L3 NO2 como segunda fuente independiente (referencia van Geffen et al. 2022).')

# Diagnóstico NO2 con el método baseline + residuo (sin kriging):
no2_runs = [r for r in loocv_results if r['gas'] == 'NO2' and r['horizonte'] == 1]
if no2_runs:
    y_true_no2 = np.concatenate([r['y_true'] for r in no2_runs])
    y_pred_no2 = np.concatenate([r['y_pred'] for r in no2_runs])
    rmse_no2 = np.sqrt(mean_squared_error(y_true_no2, y_pred_no2))
    r2_pooled_no2 = r2_score(y_true_no2, y_pred_no2)
    r2_within_no2 = float(np.mean([r2_score(r['y_true'], r['y_pred']) for r in no2_runs
                                   if len(r['y_true']) >= 2 and np.var(r['y_true']) > 1e-9]))
    print(f'\nNO2 T+1 (baseline+residuo, sin kriging): RMSE={rmse_no2:.3f} µg/m³ | '
          f'R² within-station={r2_within_no2:.3f} | R² pooled={r2_pooled_no2:.3f}')

## Bloque I — Exportación frontend: grids formato CeldaGrid

El frontend (React + Leaflet) espera grids en formato `CeldaGrid = {lat, lon, valor, varianza}`. Generamos los 9 mapas (3 gases × 3 horizontes) como JSON consumible + PNG overlays georreferenciados para `ImageOverlay` de Leaflet.

In [ ]:
# Exportación frontend — grids JSON formato CeldaGrid
FRONTEND_DIR = WORKING / 'frontend_data'
FRONTEND_DIR.mkdir(exist_ok=True)
(FRONTEND_DIR / 'overlays').mkdir(exist_ok=True)

STEP = 2  # submuestreo grilla para JSON liviano (~35x35)
grids_frontend = {}
for gas in POLLUTANTS:
    for h in HORIZONS:
        key = (gas, h)
        if key not in kriging_models:
            continue
        ok, obs = kriging_models[key]
        z_p, ss = ok.execute('grid', lon_grid, lat_grid)
        z_p = np.asarray(z_p); ss = np.asarray(ss)
        tendencia = obs.pred_cl.median()
        z_abs = z_p + tendencia
        celdas = []
        for iy in range(0, len(lat_grid), STEP):
            for ix in range(0, len(lon_grid), STEP):
                celdas.append({
                    'lat': round(float(lat_grid[iy]), 5),
                    'lon': round(float(lon_grid[ix]), 5),
                    'valor': round(float(z_abs[iy, ix]), 3),
                    'varianza': round(float(max(ss[iy, ix], 0)), 4),
                })
        grids_frontend[f'{gas}_T+{h}'] = {
            'celdas': celdas,
            'bounds': [float(lat_grid.min()), float(lon_grid.min()),
                       float(lat_grid.max()), float(lon_grid.max())],
            'vmin': round(float(z_abs.min()), 3),
            'vmax': round(float(z_abs.max()), 3),
            'n_celdas': len(celdas),
        }

with open(FRONTEND_DIR / 'grids_prediccion.json', 'w', encoding='utf-8') as f:
    json.dump(grids_frontend, f)
print(f'grids_prediccion.json: {len(grids_frontend)} mapas, {sum(len(v["celdas"]) for v in grids_frontend.values())} celdas totales')

In [ ]:
# PNG overlays georreferenciados para Leaflet ImageOverlay (transparentes, sin ejes)
for gas in POLLUTANTS:
    for h in HORIZONS:
        key = (gas, h)
        if key not in kriging_models:
            continue
        ok, obs = kriging_models[key]
        z_p, ss = ok.execute('grid', lon_grid, lat_grid)
        z_abs = np.asarray(z_p) + obs.pred_cl.median()
        sigma = np.sqrt(np.clip(np.asarray(ss), 0, None))
        for arr, suf, cmap in [(z_abs, 'pred', 'RdYlGn_r'), (sigma, 'sigma', 'magma')]:
            fig, ax = plt.subplots(figsize=(6, 6))
            ax.imshow(arr, origin='lower', cmap=cmap,
                      extent=[lon_grid.min(), lon_grid.max(), lat_grid.min(), lat_grid.max()],
                      aspect='auto')
            ax.axis('off')
            plt.savefig(FRONTEND_DIR / 'overlays' / f'{gas}_T{h}_{suf}.png',
                        dpi=100, bbox_inches='tight', pad_inches=0, transparent=True)
            plt.close()
print(f'Overlays PNG generados en {FRONTEND_DIR / "overlays"}')

## Bloque J — LISA (Local Indicators of Spatial Association)

Análisis local de Moran sobre la superficie predicha. Identifica clusters significativos de alta (HH) y baja (LL) contaminación. Cumple entregable PDF: "Análisis LISA para identificar clusters de alta y baja contaminación con significancia local".

In [ ]:
from esda.moran import Moran_Local

lisa_results = {}
for gas in POLLUTANTS:
    key = (gas, 1)
    if key not in kriging_models:
        continue
    ok, obs = kriging_models[key]
    z_p, ss = ok.execute('grid', lon_grid, lat_grid)
    z_abs = (np.asarray(z_p) + obs.pred_cl.median()).ravel()
    lon_m, lat_m = np.meshgrid(lon_grid, lat_grid)
    coords_all = np.column_stack([lon_m.ravel(), lat_m.ravel()])
    n = min(len(coords_all), 1500)
    idx = np.random.RandomState(SEED).choice(len(coords_all), n, replace=False)
    coords = coords_all[idx]; vals = z_abs[idx]
    w = DistanceBand(coords, threshold=0.015, binary=True, silence_warnings=True)
    lisa = Moran_Local(vals, w, permutations=999, seed=SEED)
    sig = lisa.p_sim < 0.05
    clusters = {
        'HH': int(((lisa.q==1)&sig).sum()), 'LL': int(((lisa.q==3)&sig).sum()),
        'HL': int(((lisa.q==4)&sig).sum()), 'LH': int(((lisa.q==2)&sig).sum()),
        'no_sig': int((~sig).sum()),
    }
    lisa_results[gas] = {'coords': coords, 'q': lisa.q, 'sig': sig, 'clusters': clusters}
    print(f'{gas} LISA T+1: {clusters}')

# Mapa LISA
n_g = len(lisa_results)
fig, axes = plt.subplots(1, n_g, figsize=(6*n_g, 6))
if n_g == 1: axes = [axes]
colores = {1:'#d7191c', 2:'#abd9e9', 3:'#2c7bb6', 4:'#fdae61', 0:'#e0e0e0'}
etiquetas = {1:'Alto-Alto', 2:'Bajo-Alto', 3:'Bajo-Bajo', 4:'Alto-Bajo', 0:'No significativo'}
for ax, (gas, res) in zip(axes, lisa_results.items()):
    coords, q, sig = res['coords'], res['q'], res['sig']
    cats = np.where(sig, q, 0)
    for cat, col in colores.items():
        m = cats == cat
        if m.any():
            ax.scatter(coords[m,0], coords[m,1], c=col, s=12, label=etiquetas[cat])
    ax.scatter(dagma_diario.longitud.unique(), dagma_diario.latitud.unique(),
               marker='X', c='black', s=80, edgecolor='white', zorder=5)
    ax.set_title(f'LISA {gas} T+1'); ax.legend(fontsize=7, loc='best'); ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(WORKING / 'lisa_clusters.png', dpi=140, bbox_inches='tight')
plt.show()

with open(FRONTEND_DIR / 'lisa.json', 'w', encoding='utf-8') as f:
    json.dump({g: r['clusters'] for g, r in lisa_results.items()}, f, indent=2)

## Bloque K — Variograma experimental + teórico ajustado

Cumple entregable PDF: "variogramas (experimental + teórico ajustado)". Muestra la semivarianza empírica y el modelo exponencial ajustado por PyKrige, con parámetros nugget/sill/range.

In [ ]:
n_g = len(POLLUTANTS)
fig, axes = plt.subplots(1, n_g, figsize=(6*n_g, 5))
if n_g == 1: axes = [axes]
variogramas = {}
for ax, gas in zip(axes, POLLUTANTS):
    key = (gas, 1)
    if key not in kriging_models:
        ax.axis('off'); continue
    ok, obs = kriging_models[key]
    try:
        lags = ok.lags
        semivar = ok.semivariance
        model_y = ok.variogram_function(ok.variogram_model_parameters, lags)
        ax.plot(lags, semivar, 'o', color='steelblue', label='experimental')
        ax.plot(lags, model_y, '-', color='red', label=f'teórico ({ok.variogram_model})')
        ax.set_title(f'Variograma {gas} T+1'); ax.set_xlabel('lag (grados)')
        ax.set_ylabel('semivarianza'); ax.legend(); ax.grid(alpha=0.3)
        p = ok.variogram_model_parameters
        variogramas[gas] = {
            'modelo': ok.variogram_model,
            'parametros': [float(x) for x in p],
            'nugget': float(p[2]) if len(p) > 2 else None,
            'sill': float(p[0]) if len(p) > 0 else None,
            'range': float(p[1]) if len(p) > 1 else None,
        }
    except Exception as e:
        ax.set_title(f'{gas}: error {e}'); ax.axis('off')
plt.tight_layout()
plt.savefig(WORKING / 'variogramas.png', dpi=140, bbox_inches='tight')
plt.show()
with open(FRONTEND_DIR / 'variogramas.json', 'w', encoding='utf-8') as f:
    json.dump(variogramas, f, indent=2)
print(json.dumps(variogramas, indent=2))

## Bloque L — K-Means: perfiles tipológicos de zonas crónicas

Cumple entregable PDF: "Análisis de perfiles tipológicos (clustering K-Means sobre las superficies predichas) para identificar zonas crónicas de alta contaminación en Cali". Cada celda de la grilla se caracteriza por su vector [NO2, SO2, O3] predicho; K-Means agrupa en perfiles.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

feature_maps = {}
for gas in POLLUTANTS:
    key = (gas, 1)
    if key in kriging_models:
        ok, obs = kriging_models[key]
        z_p, _ = ok.execute('grid', lon_grid, lat_grid)
        feature_maps[gas] = (np.asarray(z_p) + obs.pred_cl.median()).ravel()

perfiles = {}
if len(feature_maps) >= 2:
    gases_disp = list(feature_maps.keys())
    X = np.column_stack([feature_maps[g] for g in gases_disp])
    Xs = StandardScaler().fit_transform(X)
    K = 4
    km = KMeans(n_clusters=K, random_state=SEED, n_init=10).fit(Xs)
    labels = km.labels_
    lon_m, lat_m = np.meshgrid(lon_grid, lat_grid)

    fig, ax = plt.subplots(figsize=(9, 8))
    sc = ax.scatter(lon_m.ravel(), lat_m.ravel(), c=labels, cmap='Set1', s=14)
    ax.scatter(dagma_diario.longitud.unique(), dagma_diario.latitud.unique(),
               marker='X', c='black', s=120, edgecolor='white', zorder=5, label='DAGMA')
    plt.colorbar(sc, label='perfil tipológico')
    ax.set_title(f'Perfiles tipológicos K-Means (K={K}) sobre superficies T+1')
    ax.legend(); ax.set_aspect('equal')
    plt.tight_layout()
    plt.savefig(WORKING / 'perfiles_kmeans.png', dpi=140, bbox_inches='tight')
    plt.show()

    for k in range(K):
        m = labels == k
        perfil = {g: round(float(feature_maps[g][m].mean()), 3) for g in gases_disp}
        perfil['n_celdas'] = int(m.sum())
        perfil['pct_area'] = round(float(m.mean()) * 100, 1)
        # Clasificar perfil por nivel de contaminación promedio
        nivel = np.mean([feature_maps[g][m].mean() for g in gases_disp])
        perfil['interpretacion'] = 'zona crónica alta' if nivel > X.mean() else 'zona baja-media'
        perfiles[f'perfil_{k}'] = perfil
    with open(FRONTEND_DIR / 'perfiles_kmeans.json', 'w', encoding='utf-8') as f:
        json.dump(perfiles, f, indent=2)
    print(json.dumps(perfiles, indent=2))
else:
    print('Insuficientes superficies para K-Means (se requieren >=2 gases)')

## Bloque M — Estaciones + tablas para frontend

In [ ]:
# estaciones.json (formato Estacion del frontend) + loocv + kpis
est_coords = dagma_diario[['nombre_est','latitud','longitud']].drop_duplicates('nombre_est')
est_json = []
for _, r in est_coords.iterrows():
    gases = sorted(dagma_diario[dagma_diario.nombre_est == r.nombre_est].gas.str.upper().unique().tolist())
    medias = {}
    for g in gases:
        sub = dagma_diario[(dagma_diario.nombre_est == r.nombre_est) & (dagma_diario.gas.str.upper() == g)]
        medias[f'{g.lower()}_avg'] = round(float(sub.concentracion.mean()), 2)
    est_json.append({
        'id': r.nombre_est.replace(' ', '_'),
        'nombre': r.nombre_est,
        'lat': round(float(r.latitud), 5),
        'lon': round(float(r.longitud), 5),
        'contaminantes': gases,
        **medias,
    })
with open(FRONTEND_DIR / 'estaciones.json', 'w', encoding='utf-8') as f:
    json.dump(est_json, f, ensure_ascii=False, indent=2)

metricas_df.to_json(FRONTEND_DIR / 'loocv.json', orient='records')
kpis_df.to_json(FRONTEND_DIR / 'kpis.json', orient='records')

print(f'estaciones.json: {len(est_json)} estaciones')
print('\n=== Inventario frontend_data ===')
for f in sorted(FRONTEND_DIR.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(FRONTEND_DIR)}: {f.stat().st_size/1024:.1f} KB')

## Bloque N — Subir a Kaggle dataset (embeddings + frontend_data + artefactos)

Versiona `edwardsx/geovision-sit3-embeddings` con TODO: embeddings + grids + LISA + variograma + perfiles + tablas.

In [ ]:
import shutil, subprocess

DS_DIR = WORKING / 'sit3_dataset_upload'
DS_DIR.mkdir(exist_ok=True)

# Embeddings
if EMB_CACHE.exists():
    shutil.copy(EMB_CACHE, DS_DIR / 'embeddings_sit3.npz')
# frontend_data completo
shutil.copytree(FRONTEND_DIR, DS_DIR / 'frontend_data', dirs_exist_ok=True)
# Artefactos geoestadísticos
for f in ['metricas_loocv.csv','kpis_sit3.csv','mapas_predicciones.png',
          'lisa_clusters.png','variogramas.png','perfiles_kmeans.png','resumen_sit3.json']:
    src = WORKING / f
    if src.exists():
        shutil.copy(src, DS_DIR / f)

metadata = {
    'title': 'geovision-sit3-embeddings',
    'id': 'edwardsx/geovision-sit3-embeddings',
    'licenses': [{'name': 'CC0-1.0'}],
}
with open(DS_DIR / 'dataset-metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

cmd = ['kaggle','datasets','version','-p',str(DS_DIR),
       '-m','Sit3 completo: embeddings + grids frontend + LISA + variograma + K-Means','--dir-mode','zip']
res = subprocess.run(cmd, capture_output=True, text=True)
print('STDOUT:', res.stdout)
print('STDERR:', res.stderr)
if res.returncode != 0:
    print('\nSi falla version, probar create:')
    res_c = subprocess.run(['kaggle','datasets','create','-p',str(DS_DIR),'--public','--dir-mode','zip'],
                           capture_output=True, text=True)
    print(res_c.stdout, res_c.stderr)

In [ ]:
resumen = {
    'fecha_ejecucion': datetime.utcnow().isoformat() + 'Z',
    'metricas_loocv': metricas_df.to_dict(orient='records'),
    'kpis': kpis_df.to_dict(orient='records'),
    'moran_resultados': {g: {'I': float(v['I']), 'p': float(v['p']), 'EI': float(v['EI'])} for g, v in moran_results.items()},
    'bbox': list(BBOX),
    'n_secuencias_por_gas_h': {f'{g}_h{h}': len(s) for (g,h), s in secuencias_gas_h.items()},
    'limitacion_no2': 'n=2 estaciones DAGMA → LOO-CV kriging inejecutable',
}

with open(WORKING / 'resumen_sit3.json', 'w', encoding='utf-8') as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2, default=str)

print('=== ARCHIVOS PERSISTIDOS ===')
for f in sorted(WORKING.iterdir()):
    print(f'  {f.name}: {f.stat().st_size/1024:.1f} KB')

print('\n=== KPIs Sit 3 final ===')
print(kpis_df.round(3).to_string())